In [117]:
%pip install h3 ipyleaflet

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [118]:
import random
import h3
import math
from IPython.display import display
from ipywidgets import Layout, HTML

from ipyleaflet import Map, Polygon, CircleMarker, Popup, MarkerCluster, LayerGroup, GeoJSON
from typing import TypedDict, Optional

In [119]:
RESOLUTION = 8
SEARCH_LAT, SEARCH_LNG = 23.8103, 90.4125
USER_BASE_LAT_MIN = 21.5
USER_BASE_LAT_MAX = 26.3
USER_BASE_LNG_MIN = 88.5
USER_BASE_LNG_MAX = 92.2

In [120]:
class UserData(TypedDict):
    id: int
    lat: float
    lng: float
    name: str
    h3_index: str
    dot: Optional[int]


class Location(TypedDict):
    users: list[UserData]
    polygon: Optional[int]


UserStore = dict[str, Location]

In [121]:
def generate_user(i: int) -> UserData:
    lat = random.uniform(USER_BASE_LAT_MIN, USER_BASE_LAT_MAX)
    lng = random.uniform(USER_BASE_LNG_MIN, USER_BASE_LNG_MAX)

    h3_hex: str = h3.latlng_to_cell(lat, lng, RESOLUTION)

    return {
        "id": i,
        "name": f"User_{i}",
        "lat": lat,
        "lng": lng,
        "h3_index": h3_hex,
        "dot": None,
    }

In [122]:
def populate_data(n: int) -> UserStore:
    users = (generate_user(i) for i in range(n))
    data: dict[str, Location] = {}
    for u in users:
        h3_key = u['h3_index']
        if h3_key not in data:
            data[h3_key] = {"users": [], "polygon": None}

        data[h3_key]["users"].append(u)

    return data

In [123]:
def find_nearby_users(search_lat: float, search_lng: float, radius_km: int, data: UserStore) -> UserStore:
    center_hex: str = h3.latlng_to_cell(
        search_lat, search_lng, RESOLUTION)
    edge_len: float = h3.average_hexagon_edge_length(RESOLUTION, unit='km')

    k_required = math.ceil(radius_km / (edge_len * 1.5))
    search_hexes: list[str] = h3.grid_disk(center_hex, k_required)

    nearby_index: UserStore = {}

    for h_hex in search_hexes:

        cell_users = data.get(h_hex, {}).get("users", [])

        if cell_users:

            filtered_users = [
                u for u in cell_users
                if h3.great_circle_distance((search_lat, search_lng), (u['lat'], u['lng'])) <= radius_km
            ]

            if filtered_users:
                nearby_index[h_hex] = {
                    "polygon": data.get(h_hex, {}).get("polygon", None),
                    "users": filtered_users
                }

    return nearby_index

In [ ]:
def plot_indexed_users(plots: UserStore, m: Map, match: bool = False):
    main_color = "green" if match else "orange"

    def make_hex(h3_hex):
        boundary = [[lng, lat] for lat, lng in h3.cell_to_boundary(h3_hex)]
        boundary.append(boundary[0])
        return {
            "type": "Feature",
            "properties": {
                "type": "hexagon",
                "style": {"color": main_color, "fillColor": main_color, "fillOpacity": 0.1, "weight": 1}
            },
            "geometry": {"type": "Polygon", "coordinates": [boundary]}
        }

    def make_user(user, h3_hex):
        return {
            "type": "Feature",
            "properties": {
                "type": "user_point",
                "popup_content": f'<div style="font-family:sans-serif;padding:5px;"><strong style="color:{main_color};">User ID:</strong> {user["id"]}<br><strong>H3:</strong> <code>{h3_hex}</code></div>',
                "style": {"radius": 5, "color": main_color, "fillColor": main_color, "fillOpacity": 0.7}
            },
            "geometry": {"type": "Point", "coordinates": [user['lng'], user['lat']]}
        }

    features = [make_hex(h3_hex) for h3_hex in plots] + \
               [make_user(u, h3_hex) for h3_hex, loc in plots.items()
                for u in loc["users"]]

    full_geojson_data = {"type": "FeatureCollection", "features": features}

    def on_click_handler(feature, **kwargs):
        if feature.get('properties', {}).get('type') != 'user_point':
            return

        lng, lat = feature['geometry']['coordinates']
        m.add_layer(Popup(
            location=(lat, lng),
            child=HTML(value=feature['properties']['popup_content']),
            name="Popup"
        ))

    user_layer = GeoJSON(
        data=full_geojson_data,
        name="UserMap",
        style_callback=lambda f: f['properties']['style'],
        point_style={"radius": 5, "fillOpacity": 0.7}
    )
    user_layer.on_click(on_click_handler)
    m.add_layer(user_layer)

In [125]:
map_layout = Layout(width='100vw', height='700px')
display_map = Map(center=(SEARCH_LAT, SEARCH_LNG), zoom=9, prefer_canvas=True,
                  scroll_wheel_zoom=True, layout=map_layout)
display(display_map)

Map(center=[23.8103, 90.4125], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zo…

In [126]:
search_data = populate_data(100_000)

plot_indexed_users(search_data, display_map, match=False)

radius = 10

results = find_nearby_users(SEARCH_LAT, SEARCH_LNG, radius, search_data)

count = sum(len(location["users"]) for location in results.values())

plot_indexed_users(results, display_map, True)

print(f"Found {count} users within {radius}km of Dhaka center.")

Found 164 users within 10km of Dhaka center.
